# 03 — 필수 유전자 이진 분류 (`results/esscls`)

> **GPU 상주 판** — `03-train_esscls.ipynb` 와 같은 데이터 · 폴드 분할 · 모델 · 학습 · 평가 규약에서 **ESM 토큰 임베딩까지 모든 입력을 GPU 메모리에 올려** 배치를 GPU 안에서 색인으로 만든다. 배치 공급 스레드와 pinned 버퍼가 없다. VRAM 180 GB급 GPU(예: B200)용이며, ESM 토큰은 bf16 으로 약 145 GiB를 차지한다. 결과 폴더는 원본 노트북과 같다.

유전자 결손 시 **필수(1)** 인지 **비필수(0)** 인지 예측한다. genus 단위 중첩 교차검증으로 새 genus에 대한 성능을 평가하고,
전체 샘플로 최종 모델을 훈련한다.

| `fit_mean` | 라벨 | 샘플 수 |
|---|---|---|
| `-inf` | **1** (필수) | 18,037 |
| 수치 | **0** (비필수) | 177,317 |
| 공란 (미상) | 제외 | 22,114 |

46균주 · 28 genus · 195,354 샘플, 양성 9.23%. 필수 샘플이 없는 균주는 훈련에만 쓰고 검증 genus 후보와 지표 총계에서는 뺀다.

진행: 1 데이터 → 2 폴드 분할 → 3 모델 · 손실 · 지표 → 4 중첩 교차검증(성능 평가) → 5 최종 모델

In [1]:
# !pip install torch scikit-learn h5py pandas matplotlib tqdm biopython
# !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [2]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # 배치 모양이 계속 바뀌어도 VRAM 조각화를 줄인다 (torch 불러오기 전)
import time, numpy as np, pandas as pd, torch
from proteomefit import batching, data, features, folds, cv
from proteomefit.config import Config, setup_torch
from proteomefit.models import ds_blocks, param_summary
from proteomefit.tasks import EssTask

cfg = Config(
    out_dir    = '../results/esscls_50',
    droprate   = 0.5,                        # 연결 벡터 · MLP 은닉층 · PMA 드롭아웃
    mlp_hidden = (640, 320),                 # MLP 은닉 유닛 수 (길이 = 은닉층 수)
    lr         = 1e-3,                       # 학습률 (AdamW)
    min_delta  = 0.01,                       # 검증 MCC 개선 인정 하한
)                                            # 나머지 설정 = proteomefit/config.py 기본값
INPUTS = ('seq', 'esm', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb')   # 전체 입력 (연결 순서)
setup_torch()

print('device:', cfg.device, f'| {cfg.inner_k} inner folds (val genus matched to test extrapolation distance) '
      '| loss weighted BCE | early stop on val MCC')
print(f'classes: {EssTask.classes} | evaluated on all samples | threshold {cfg.thresh}')
print('autocast:', cfg.amp_dtype, '| compile:', cfg.compile, '| all inputs GPU-resident')
if cfg.device != 'cpu':
    p = torch.cuda.get_device_properties(0)
    print(f'gpu: {p.name} | VRAM {p.total_memory / 2**30:.0f} GiB | torch {torch.__version__}')

device: cuda:0 | 3 inner folds (val genus matched to test extrapolation distance) | loss weighted BCE | early stop on val MCC
classes: ('non-essential', 'essential') | evaluated on all samples | threshold 0.5
autocast: torch.bfloat16 | compile: True | all inputs GPU-resident
gpu: NVIDIA B200 | VRAM 178 GiB | torch 2.8.0+cu128


## 1. 데이터

모델 입력은 가변 길이 입력 2종과 고정 길이 **데이터셋** 1종이다.

### 1.1 가변 길이 입력 — 각각 PMA로 벡터 하나로 축약

| 이름 | 형태 | 출처 | 내용 |
|---|---|---|---|
| ESM 토큰 임베딩 | `[T, 1152]` | `emb_esm_600m.h5` | ESM-C 600M 토큰별 표현 (BOS/EOS 포함) |
| 정수 인코딩 서열 | `[L]` | `seq_aa` | 0 = 패딩, 1 ~ 20 = 아미노산, 21 = 그 외. 앞 `MAX_LEN` 잔기 |

### 1.2 데이터셋 — 샘플 × 1496열

| 블록 | 열 | 내용 |
|---|---|---|
| `prot` | 1152 | ProteomeLM 임베딩, 균주 단위 z-정규화 |
| `bio` | 140 | Biopython 서열 특성 (물리화학 12 · 조성 20 · N말단 20 · C말단 20 · 코돈 빈도 64 · GC 4), 균주 단위 z-정규화 |
| `ctx` | 68 | 문맥 특성, ESM 평균 임베딩 유사도 공간 |
| `ctxp` | 68 | 문맥 특성, ProteomeLM 임베딩 유사도 공간 |
| `ctxb` | 68 | 문맥 특성, 두 공간을 이은 유사도 공간 |

**문맥 특성** — 질의 샘플마다 46개 균주 각각에서 가장 유사한 단백질 1개를 찾고, 그중 **참조 균주** 의 것만
유사도 상위 $k \in \{1, 3, 5, 10, 20\}$ 개로 요약한다. 참조 균주는 **시험 genus · 검증 genus · 자기 genus
어디에도 속하지 않는 균주** 이므로, 데이터셋은 내부 폴드마다 다시 만든다.

| 묶음 | 열 | 내용 |
|---|---|---|
| `cons_*` | 9 | 보존 정도 — 상위 $k$ 평균 코사인, 1위와 3위 · 10위의 격차, 코사인 표준편차, 참조 수 (라벨 미사용) |
| `otf_*` | 25 | 참조 단백질의 적합도 · 필수 여부 요약 (원본 척도) |
| `otr_*` | 12 | 같은 요약, 균주 내 순위 척도 |
| `rbh_*` | 14 | 상호 최적 히트만으로 한 요약 |
| `para_*` | 8 | 자기 게놈 안 유사 단백질의 코사인 요약 (라벨 미사용) |

In [3]:
# 라벨·메타 - fit_mean 이 공란(미상)인 샘플만 제외 (n_all 217,468 -> n 195,354)
D    = data.load_samples(cfg.base_dir, cfg.label_col, finite_only=False)
task = EssTask(D, cfg)                                          # 라벨 y [n] (1 = 필수) · GPU 라벨
y    = task.y

n_pos_org = pd.Series(y).groupby(D.org).sum()
NO_ESS    = n_pos_org.index[n_pos_org == 0].to_numpy()          # 필수 샘플이 없는 균주
print(f'samples:{len(D.lab):,} / {len(D.base):,} (dropped unknown {len(D.base) - len(D.keep):,})')
print(f'  essential {int(y.sum()):,} ({y.mean() * 100:.2f}%) | non-essential {int((y == 0).sum()):,}')
print(f'no essential samples: {len(NO_ESS)} organisms ({", ".join(map(str, NO_ESS))})')

samples:195,354 / 217,468 (dropped unknown 22,114)
  essential 18,037 (9.23%) | non-essential 177,317
no essential samples: 4 organisms (Burk376, Magneto, Putida, RalstoniaUW163)


In [4]:
# Biopython 특성 - 전 유전자에서 계산 -> 균주별 z-정규화 -> 사용 샘플만 [n, 140]
seq_aa_all = D.base.seq_aa.fillna('').to_numpy()
BIOF  = features.bio_features(seq_aa_all, D.base.seq_dna.fillna('').to_numpy())   # [n_all, 140]
D.BIO = features.zscore_per_org(BIOF, D.org_code_all)[D.keep]
del BIOF

# 정수 인코딩 서열 - 앞 max_len 잔기를 한 배열에 이어 붙인다 (샘플 i = seq_flat[seq_off[i]:seq_off[i+1]], 훈련 때 배열째 GPU 상주)
D.seq_flat, D.seq_off, D.seq_len, raw_len = features.encode_seqs(seq_aa_all[D.keep], cfg.max_len)
print(f'seq residues {len(D.seq_flat):,} (max {int(D.seq_len.max())}, truncated {int((raw_len > cfg.max_len).sum())})')
print('bio', D.BIO.shape)

biopython: 100%|██████████| 217468/217468 [00:37<00:00, 5774.74it/s]


seq residues 66,920,905 (max 4096, truncated 80)
bio (195354, 140)


### 직교상동체 목록

전체 217,468개 단백질(라벨 미상 포함) 각각에 대해, 46개 균주마다 코사인이 가장 높은 단백질의 번호와 그 값을
구한다(`ORT`). 상호 최적 히트를 판정하려면 역방향 조회가 필요해 전체 단백질을 쓴다. 임베딩은 차원별 표준화 후
L2 정규화(백색화)해 내적이 곧 코사인이 되게 한다. 같은 셀에서 균주 프로테옴 표현 `ORG_VEC`(2절)도 구한다.

In [5]:
prot_raw, esm_mean_raw = data.load_embeddings(cfg.prot_h5, cfg.mean_h5, D)        # [n_all, 1152] x 2
D.PROT    = features.zscore_per_org(prot_raw, D.org_code_all)[D.keep]             # 데이터셋 prot 블록 [n, 1152]
D.ORG_VEC = pd.DataFrame(prot_raw).groupby(D.org_code_all).mean().to_numpy()      # 균주 프로테옴 표현 [n_org, 1152]

# 직교상동체 목록 3종 (ESM · ProteomeLM · 둘을 이은 공간) - 값 = (sim, idx, para)
D.ORT = features.ortholog_lists(esm_mean_raw, prot_raw, D.org_code_all, D.n_org, max(cfg.topk), cfg.sim_chunk, cfg.device)
del prot_raw, esm_mean_raw
print('ortholog lists:', {k: (v[0].shape, v[2].shape) for k, v in D.ORT.items()})

para both: 100%|██████████| 46/46 [00:00<00:00, 347.44it/s]


ortholog lists: {'esm': ((217468, 46), (217468, 20)), 'prot': ((217468, 46), (217468, 20)), 'both': ((217468, 46), (217468, 20))}


In [6]:
# ESM 토큰별 임베딩 - 사용 샘플의 전 토큰을 GPU 한 배열에 이어 붙인다 (샘플 i = esm[esm_off[i]:esm_off[i+1]], bf16)
D.esm, D.esm_off, D.esm_len = data.load_esm_tokens(cfg.esm_h5, D, cfg.device, cfg.amp_dtype, cfg.esm_chunk, cfg.gpu_reserve)
print(f'ESM tokens:{len(D.esm):,} x {D.esm.shape[1]} = {D.esm.nbytes / 2**30:.1f} GiB GPU (bf16) | '
      f'free after load {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')
print(f'tokens per sample: min {D.esm_len.min()} med {int(np.median(D.esm_len))} max {D.esm_len.max()}')

ESM -> GPU: 100%|██████████| 107/107 [03:21<00:00,  1.88s/it]

ESM tokens:67,311,613 x 1152 = 144.4 GiB GPU (bf16) | free after load 33.1 GiB
tokens per sample: min 12 med 298 max 4098


In [7]:
D.block_dim = features.block_dims(D, cfg.seq_dim)                # 입력별 연결 벡터 폭
blocks = ds_blocks(INPUTS)                                       # 데이터셋 블록
print('dataset blocks: ' + ' + '.join(f'{b}({D.block_dim[b]})' for b in blocks) + f' = {sum(D.block_dim[b] for b in blocks)}')
print(f'organisms:{D.lab.org_id.nunique()} | genus:{D.lab.genus.nunique()}')
sh = pd.Series(y).groupby(D.gen).mean()                          # genus별 필수 비율 [n_fold]
print(f'per-genus essential share (min~max): {sh.min():.3f}~{sh.max():.3f}')

dataset blocks: prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68) = 1496
organisms:46 | genus:28
per-genus essential share (min~max): 0.000~0.244


## 2. 폴드 분할 — 외부 · 내부 모두 genus 단위

$$\text{test}_{k} = G_k, \qquad \text{val}_{k,j} = G_{v_{k,j}}, \qquad \text{train}_{k,j} = \{\, i \mid \mathrm{gid}(i) \notin \{k,\ v_{k,j}\} \,\}$$

- $G_g$ — genus $g$ 의 전 샘플
- **외부 폴드** $k$ — 테스트 genus $k$ (28개, genus 단위 리브-원-아웃 교차검증(Leave-One-Out Cross-Validation, LOOCV))
- **내부 폴드** $j$ — 검증 genus $v_{k,j}$ 하나 (`INNER_K` = 3개). 검증 genus는 훈련에서 통째로 빠진다
- 내부 폴드는 **성능 평가 전용** 이다. 내부 폴드 모델 3개를 각각 테스트 genus에 평가한 지표의 평균이 그 genus의 성능이고, 실제 예측에는 5절의 최종 모델을 쓴다

### 검증 genus 선택 — 시험 genus와 외삽 거리가 비슷한 genus

$$\mathbf{u}_o = \frac{1}{n_o} \sum_{i \in o} \mathbf{e}_i, \qquad \mathbf{v}_g = \frac{1}{\lvert O_g \rvert} \sum_{o \in O_g} \mathbf{u}_o, \qquad s(k, g) = \cos\big(\mathbf{v}_k - \bar{\mathbf{v}},\ \mathbf{v}_g - \bar{\mathbf{v}}\big)$$

- $\mathbf{e}_i$ — 유전자 $i$ 의 ProteomeLM 임베딩 (z-정규화 전 원본)
- $\mathbf{u}_o$ — 균주 $o$ 의 프로테옴 표현 = 그 균주 전 유전자 $n_o$ 개의 평균 (`ORG_VEC`)
- $\mathbf{v}_g$ — genus $g$ 의 표현 = 그 genus 균주 집합 $O_g$ 의 $\mathbf{u}_o$ 평균 (`GVEC`)
- $\bar{\mathbf{v}}$ — 전 genus 표현의 평균. 공통 성분을 빼고 코사인을 잰다
- $s(k, g)$ — 시험 genus $k$ 와 genus $g$ 의 코사인 유사도 (`GSIM`)

$$s^{\mathrm{nn}}(g \mid X) = \max_{h \notin X \cup \{g\}} s(g, h), \qquad d(v) = \big\lvert s^{\mathrm{nn}}(v \mid \{k\}) - s^{\mathrm{nn}}(k \mid \varnothing) \big\rvert$$

- $s^{\mathrm{nn}}(g \mid X)$ — genus $g$ 와, $X$ 를 뺀 나머지 genus 중 가장 닮은 것의 유사도 = **외삽 거리** (클수록 가까운 친척이 훈련에 있다, `nn_sim`)
- $s^{\mathrm{nn}}(k \mid \varnothing)$ — 시험 genus 가 풀 외삽, $s^{\mathrm{nn}}(v \mid \{k\})$ — 검증 genus $v$ 가 내부 폴드에서 풀 외삽 (시험 genus 도 훈련에서 빠지므로 제외)
- 후보를 $d(v)$ 오름차순으로 세워 앞의 `INNER_K` 개를 검증 genus 로 쓴다 — 검증 문제의 난이도를 시험 문제에 맞춘다
- 시험 genus 의 최근접 genus 는 후보에서 빼 훈련 · 문맥 참조에 남긴다
- 예: 시험 Pseudomonas 의 $s^{\mathrm{nn}} = 0.39$ → $s^{\mathrm{nn}} = 0.41$ 인 Acidovorax 는 $d = 0.02$ 로 뽑히고, 0.89 인 Klebsiella ($d = 0.50$) 는 뽑히지 않는다
- 후보 = 시험 genus와 그 최근접 genus를 뺀 genus 중 검증 샘플에 양·음성이 모두 있는 균주가 하나 이상인 것 (검증 MCC가 정의되도록). 필수 샘플이 없는 `Paraburkholderia` · `Magnetospirillum` 은 후보에서 빠진다

In [8]:
# genus 표현 -> 중심화 코사인 유사도 -> 검증 genus 선택 (후보 = 검증 샘플에 양·음성이 모두 있는 균주가 하나 이상인 genus)
S = folds.make_splits(D.gen, D.org2genus, D.ORG_VEC, cfg.inner_k, candidate=task.val_candidate)
folds.print_splits(S, D.lab, cfg.inner_k, y=y)                   # 분할 요약 + 누출 점검 (warn 이 없으면 정상)

nested CV: 28 outer x 3 inner (genus-level) = 84 trainings / 195,354 samples
val-genus candidates 26/28 (excluded: Paraburkholderia, Magnetospirillum)

fold01 Pseudomonas     : test 48,716(10org, pos  7.5%) | val Acidovorax(+0.17, nn 0.41, 4,428) Desulfovibrio(-0.32, nn 0.42, 5,947) Marinobacter(+0.13, nn 0.42, 4,203)
fold02 Ralstonia       : test 16,451(4org, pos  8.4%) | val Azospirillum(+0.11, nn 0.75, 4,518) Dickeya(-0.10, nn 0.76, 11,870) Shewanella(-0.50, nn 0.61, 14,788)
fold03 Shewanella      : test 14,788(4org, pos  9.4%) | val Sphingomonas(-0.23, nn 0.60, 3,803) Caulobacter(-0.04, nn 0.60, 3,745) Dyella(-0.41, nn 0.58, 3,982)
fold04 Dickeya         : test 11,870(3org, pos 10.3%) | val Azospirillum(-0.27, nn 0.75, 4,518) Ralstonia(-0.10, nn 0.69, 16,451) Cupriavidus(-0.18, nn 0.69, 6,847)
fold05 Cupriavidus     : test  6,847(1org, pos  6.9%) | val Azospirillum(-0.01, nn 0.75, 4,518) Dickeya(-0.18, nn 0.76, 11,870) Shewanella(-0.39, nn 0.61, 14,788)
fold06 Desulfovibrio   : tes

## 3. 모델 · 손실 · 지표

### 구조

```
정수 서열 [B,L] ─── 임베딩 + 위치 ─ PMA ── [B,64] ───┐
ESM 토큰 [B,T,1152] ── PMA ────────────── [B,1152] ─┼─ concat [B,2712] ─ MLP ─ 로짓 [B]
데이터셋 [B,1496] ───────────────────────────────────┘
```

- 다중 헤드 어텐션 풀링(Pooling by Multihead Attention, PMA) — 학습 가능한 시드 벡터 1개를 쿼리로, 토큰열을 어텐션 가중 평균해 벡터 하나로 줄인다. 쿼리가 입력과 무관하므로 키 투영은 쿼리 쪽으로 접고 값 투영은 가중 평균 뒤로 미뤄, 토큰마다 1152 → 2304 투영을 하지 않는다(결과는 `nn.MultiheadAttention` 과 같다)
- 다층 퍼셉트론(Multi-Layer Perceptron, MLP) — `Dropout → [Linear → GELU → Dropout] × len(MLP_HIDDEN) → Linear(1)`. `MLP_HIDDEN = [640, 320]` 이면 2712 → 640 → 320 → 1 이다. 출력이 로짓 $z$ 이고 $\sigma(z)$ 가 필수일 확률이다

### 손실 — 가중 이진 교차 엔트로피(Binary Cross-Entropy, BCE)

$$\mathcal{L} = \frac{1}{\sum_i w_i} \sum_i w_i \Big[ -\,p_{+}\, y_i \log \sigma(z_i) - (1 - y_i) \log\big(1 - \sigma(z_i)\big) \Big], \qquad w_i = \frac{1}{n_{\mathrm{org}}(g_i)\, n_{\mathrm{smp}}(o_i)}$$

- $y_i$ — 라벨 (1 = 필수), $z_i$ — 로짓
- $p_{+}$ — 양성 가중 = 훈련 샘플의 음성 수 / 양성 수 (전체 기준 $177{,}317 / 18{,}037 \approx 9.8$)
- $w_i$ — 분류군 역크기 가중치. $n_{\mathrm{org}}(g_i)$ = 샘플 $i$ 의 genus 안 균주 수, $n_{\mathrm{smp}}(o_i)$ = 그 균주의 샘플 수. 훈련 샘플 평균 1로 정규화한다
- 예: 균주 2개인 genus에서 4,000 샘플 균주의 $w_i = 1/8000$ → 그 균주 합 $1/2$, genus 합 1 (genus마다 같은 비중)

### 평가 샘플

테스트 genus의 **전 샘플** 로 평가한다. 검증셋은 검증 genus의 전 샘플이다.

### 평가 지표 — 매튜스 상관계수(Matthews Correlation Coefficient, MCC)

$$MCC = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

- 분자 — 맞춘 조합과 틀린 조합의 곱의 차. 모두 한 클래스로 예측하면 0
- 분모 — 예측 양성 · 실제 양성 · 예측 음성 · 실제 음성 수의 곱의 제곱근. 값을 $[-1, 1]$ 로 맞춘다
- 예: $TP=60,\ FN=40,\ FP=90,\ TN=810$ → 분자 $45{,}000$, 분모 $\approx 107{,}121$ → $MCC \approx 0.42$

MCC는 균주별로 구해 평균한다. 양·음성이 모두 있는 균주만 넣는다(`org_masks` · `in_total`). 조기 종료와 결과 정렬의
기준은 MCC이고, 결과 파일에는 혼동행렬 4칸 · F1 · 재현율 · 정밀도 · 정확도 · ROC AUC · PR AUC 도 함께 남긴다.

### 구현 위치

| 내용 | 모듈 |
|---|---|
| 문맥 특성 · 데이터셋 (`ctx_block` · `build_dataset`) | `proteomefit/features.py` |
| 모델 (`PMA` · `SeqBranch` · `EssNet`) | `proteomefit/models.py` |
| 배치 구성 · GPU 상주 배치 공급 · 추론 | `proteomefit/batching.py` |
| 라벨 · 손실 · 샘플 가중치 · 조기 종료 지표 · 평가 산출물 (`EssTask`) | `proteomefit/tasks.py` |
| 지표 · 그림 | `proteomefit/metrics.py` · `proteomefit/plots.py` |
| 훈련 루프 (`train_model`) | `proteomefit/train.py` |
| 중첩 교차검증 · 최종 모델 · 요약 (`run_cv` · `train_final` · `print_summary`) | `proteomefit/cv.py` |

## 4. 중첩 교차검증 — 성능 평가

내부 폴드 $(k, j)$ 마다 데이터셋 생성(시험 · 검증 genus를 참조에서 제외, 훈련 샘플 기준 표준화) → 훈련(검증 MCC 조기 종료)
→ best epoch 가중치로 **테스트 genus $k$ 의 전 샘플** 평가 순으로 진행한다.

| 경로 | 산출물 |
|---|---|
| `fold##_<genus>/inner##/` | `model.pt` · `history.csv` · `history.png` · `metrics_organ.csv` · `confusion.png` · `pred.npz` (평가 샘플 필수 확률 `prob`, 테스트 genus 샘플 순) |
| `fold##_<genus>/` | `metrics_folds.csv` |
| `OUT_DIR/` | `metrics_total.csv` · `model_final.pt` · `history_final.csv` · `history_final.png` |

- `metrics_organ.csv` — 테스트 균주별 1줄. 혼동행렬 4칸 + MCC · F1 · 재현율 · 정밀도 · 정확도 · ROC AUC · PR AUC
- `metrics_folds.csv` — 내부 폴드별 1줄 (검증 genus · 시험 genus 와의 유사도 `val_sim` · 외삽 거리 `val_nn` · 균주 평균 지표, `_sd` = 균주 간 표준편차) + mean · median · std 줄
- `metrics_total.csv` — genus별 1줄 (내부 폴드 지표의 평균. best epoch 은 중앙값 `best_epoch_med` (채택) · 평균 `best_epoch_mean` 을 함께 남긴다, `_sd` = 내부 폴드 간 표준편차) + genus 간 mean · median · std 줄

`RESUME=True` 면 산출물 6종이 모두 있는 내부 폴드는 저장된 csv 를 읽고 넘어간다.

In [13]:
os.makedirs(cfg.out_dir, exist_ok=True)
_m = task.build_model(INPUTS, cfg.seed)
print(f'model params {param_summary(_m)} | {task.arch_msg(_m)}')
print(f'DROPRATE={cfg.droprate} | COMPILE={cfg.compile} | {S.n_fold * cfg.inner_k:,} trainings')
del _m

model params 7.54M = PMA 5.32M + seq 0.28M + MLP 1.94M | MLP 2712 -> 640 -> 320 -> 1
DROPRATE=0.5 | COMPILE=True | 84 trainings


In [14]:
t_start = time.perf_counter()
res = cv.run_cv(task, D, S, cfg, INPUTS, cfg.out_dir)            # genus별 결과 [n_fold, *] (metrics_total.csv 저장)


=== fold 1/28  Pseudomonas === test 48,716 samples (10 org, pos 7.5%) | 3 cached
  inner01 val Acidovorax       (sim +0.17, nn 0.41) best@9    val_mcc=0.725 | test mcc=0.705 recall=0.809 precision=0.664 roc_auc=0.955 pr_auc=0.835 | cached
  inner02 val Desulfovibrio    (sim -0.32, nn 0.42) best@10   val_mcc=0.604 | test mcc=0.694 recall=0.838 precision=0.624 roc_auc=0.957 pr_auc=0.838 | cached
  inner03 val Marinobacter     (sim +0.13, nn 0.42) best@17   val_mcc=0.747 | test mcc=0.716 recall=0.782 precision=0.704 roc_auc=0.952 pr_auc=0.832 | cached
  -------------------------------------------------------------------------------------------------------------------------------
  fold 1/28 Pseudomonas (0:00:00) | best@10.0±4.4 val_mcc=0.692±0.077 | mcc=0.705±0.011 recall=0.810±0.028 precision=0.664±0.040 roc_auc=0.955±0.002 pr_auc=0.835±0.003

=== fold 2/28  Ralstonia === test 16,451 samples (4 org, pos 8.4%) | 3 cached
  inner01 val Azospirillum     (sim +0.11, nn 0.75) best@34   val_m

In [15]:
cv.print_summary(task, res, S.n_fold, cfg.inner_k, time.perf_counter() - t_start)
print(f'\nsaved {cfg.out_dir}/metrics_total.csv | fold##_<genus>/metrics_folds.csv | fold##_<genus>/inner##/')


SUMMARY  28 genus x 3 inner folds | total elapsed 0:00:04
  #  genus                test    best@       val_mcc         mcc          recall      precision      roc_auc        pr_auc   
------------------------------------------------------------------------------------------------------------------------------
  1  Pseudomonas        48,716   10.0±4.4    0.692±0.077   0.705±0.011   0.810±0.028   0.664±0.040   0.955±0.002   0.835±0.003
  2  Ralstonia          16,451   34.0±10.2   0.729±0.044   0.779±0.007   0.800±0.006   0.805±0.017   0.955±0.002   0.868±0.006
  3  Shewanella         14,788   25.0±6.1    0.730±0.020   0.722±0.005   0.849±0.012   0.664±0.015   0.956±0.001   0.850±0.001
  4  Dickeya            11,870   27.0±6.1    0.743±0.055   0.768±0.009   0.817±0.010   0.770±0.025   0.960±0.002   0.874±0.002
  5  Cupriavidus         6,847   23.0±11.0   0.722±0.036   0.752±0.013   0.842±0.010   0.706±0.028   0.969±0.006   0.870±0.005
  6  Desulfovibrio       5,947   15.0±1.5    0.742±0

## 5. 최종 모델

전체 샘플로 훈련한 배포용 모델이다. 검증셋이 없으므로 epoch 수는 4절의 best epoch으로 정한다.

$$\texttt{N\_EPOCH\_FINAL} = \big\lfloor \mathrm{median}_k\, \mu_k + 0.5 \big\rfloor, \qquad \mu_k = \mathrm{median}_j\, e_{k,j}$$

- $e_{k,j}$ — genus $k$ 의 내부 폴드 $j$ 에서 조기 종료가 고른 best epoch
- $\mu_k$ — genus $k$ 의 best epoch = 내부 폴드 3개의 중앙값 (`metrics_total.csv` 의 `best_epoch_med`, 평균은 `best_epoch_mean`). 폴드 수가 홀수라 정수다
- $\mathrm{median}_k\, \mu_k$ — genus 28개의 중앙값. 개수가 짝수라 가운데 두 값의 평균이 되어 .5 가 나올 수 있으므로 반올림한다
- 중앙값을 쓰면 조기 종료가 유난히 늦거나 이른 폴드 · genus 하나가 epoch 수를 끌고 가지 않는다
- 예: 내부 폴드 best epoch (22, 30, 31, 35, 90) → $\mu_k = 31$ (평균이면 41.6). genus 값이 (30, 33, 38, 41) 이면 중앙값 35.5 → 36 epoch

문맥 특성은 자기 genus만 참조에서 빼고 만들며(`build_dataset(-1, -1, ·)`), 표준화 기준도 전체 샘플이다.

In [ ]:
model = cv.train_final(task, D, S, cfg, INPUTS, cfg.out_dir, res)   # 전체 샘플 최종 모델 -> model_final.pt
print(f'total elapsed {cv.hms(time.perf_counter() - t_start)}')

final training epochs = 23 (median of genus medians 23.0 | mean of genus means 22.7)


final:   0%|          | 0/23 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7095: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
final:  26%|██▌       | 6/23 [00:13<00:21,  1.28s/it, loss=0.5052/nan MCC=0.634/nan best=-2.000@0 pat=0]